# Facilitator-Member Difference Identification

## Creating the csv file for the following:

`Name-conference-year`

**Also for the following:**

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

In [ ]:
# ==================== NICO person-level pipeline with ANNOTATIONS ====================
import json, re
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# ---- CONFIG ----
# DATA_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")
# OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
DATA_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PERSON_SESSION = []   # collect across conferences
ALL_PERSON_YEAR    = []   # collect across conferences

# ---- HELPERS ----
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def _norm_code_name(name: str) -> str:
    """
    Normalize annotation code names to snake-case tokens for column names.
    E.g., "Knowledge Sharing" -> "knowledge_sharing"
          "Coordination and Decision Practices" -> "coordination_decision_practices"
    """
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = "_".join(s.split())
    return s

# ---- LOAD CONFERENCE BUNDLE ----
def load_conference_data(conf_path: Path):
    conf_name   = conf_path.name   # e.g., "2021MZT"
    year        = int(conf_name[:4])
    conference  = conf_name[4:]

    outcome          = _load_json(conf_path / f"{conf_name}_outcome.json")
    person_to_team   = _load_json(conf_path / f"{conf_name}_person_to_team.json")
    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (per session)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")      # e.g., "2021_09_30_MZT_S5"
        features[sid] = _load_json(fp)

    return year, conference, outcome, person_to_team, session_outcomes, features

# ---- PERSON METRICS FROM TRANSCRIPTS + ANNOTATIONS (session_data/*.json) ----
def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Returns dict: { person_name -> { p_* metrics, ann_* metrics } } for one session.
    - p_*: speaking, turns, interruptions, overlaps, screenshares, smiles, nods
    - ann_*: for each annotation code name:
        ann_<code>_count, ann_<code>_sum_score, ann_<code>_mean_score
    """
    session_file = conf_path / "session_data" / f"{session_id}.json"
    if not session_file.exists():
        return {}

    data = _load_json(session_file)
    rows = data.get("all_data", [])
    # Counters
    by_person_counts = defaultdict(lambda: Counter())  # p_* tallies
    # For annotations, we need per code: count, sum of scores -> then derive mean
    by_person_ann_count = defaultdict(lambda: Counter())
    by_person_ann_sum   = defaultdict(lambda: Counter())

    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue

        # --- p_* metrics ---
        dur = r.get("speaking_duration", 0) or 0
        by_person_counts[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person_counts[speaker]["p_turns"] += 1

        if str(r.get("interuption", "")).strip().lower() == "yes":
            by_person_counts[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap", "")).strip().lower() == "yes":
            by_person_counts[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare", "")).strip().lower() == "yes":
            by_person_counts[speaker]["p_screenshare_segments"] += 1

        by_person_counts[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person_counts[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person_counts[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

        # --- ann_* metrics ---
        ann = r.get("annotations") or {}
        if isinstance(ann, dict):
            for raw_code, payload in ann.items():
                code = _norm_code_name(raw_code)
                # Count one utterance carrying this code
                by_person_ann_count[speaker][f"ann_{code}_count"] += 1
                # Pull "score" if present
                score = None
                if isinstance(payload, dict):
                    score = payload.get("score", None)
                if score is not None:
                    try:
                        s_val = float(score)
                    except Exception:
                        s_val = None
                    if s_val is not None:
                        by_person_ann_sum[speaker][f"ann_{code}_sum_score"] += s_val

    # Build person metrics dict (combine p_* and ann_*)
    out = {}
    for person in set(list(by_person_counts.keys()) + list(by_person_ann_count.keys())):
        d = dict(by_person_counts[person])

        # Merge ann counts + sums and compute means where possible
        for k, v in by_person_ann_count[person].items():
            d[k] = float(v)  # count
            # derive code token
            code = k.replace("ann_", "").replace("_count", "")
            sum_key = f"ann_{code}_sum_score"
            mean_key = f"ann_{code}_mean_score"
            ssum = float(by_person_ann_sum[person].get(sum_key, 0.0))
            d[sum_key] = ssum
            # mean only if count > 0 and we observed any score
            if v > 0 and ssum > 0:
                d[mean_key] = ssum / v
            else:
                # not all annotations have scores; keep NaN to avoid bias
                d[mean_key] = float("nan")
        out[person] = d

    return out

# ---- BUILD person-session rows ----
def build_person_session(year, conference, conf_path, session_outcomes, features):
    """One row per (person, session): role_in_session, ctx_* session features, p_* + ann_* metrics."""
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]
    rows = []

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        # union of all team members in 'teams'
        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = facilitators | speakers | members

        # session-level features (context -> ctx_*)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}

        # person-level from transcripts + annotations
        person_metrics = extract_person_metrics_from_session_data(conf_path, sid)

        for person in sorted(people):
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            pmet = person_metrics.get(person, {})  # p_* + ann_* keys
            rows.append({
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx,
                **pmet
            })

    return pd.DataFrame(rows)

# ---- BUILD person-year (aggregate) ----
def build_person_year(person_session_df, person_to_team):
    """
    Aggregate to one row per (person, conference, year):
      - role tallies & primary role
      - mean of ctx_*, p_*, ann_* across sessions attended
      - team outcomes joined by person
    """
    if person_session_df.empty:
        return person_session_df

    # Role tallies
    pivot = (person_session_df
             .pivot_table(index=["person_name","conference","year"],
                          columns="role_in_session",
                          values="session_id",
                          aggfunc="nunique",
                          fill_value=0)
             .reset_index())
    for col in ["facilitator","member","participant","unknown"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["sessions_total"] = pivot["facilitator"] + pivot["member"] + pivot["participant"] + pivot["unknown"]

    # Primary role: facilitator > member > participant > unknown
    role_priority = {"facilitator": 3, "member": 2, "participant": 1, "unknown": 0}
    def primary_role(row):
        # choose highest priority among any nonzero counts
        best = max(role_priority, key=lambda r: (row.get(r,0)>0, role_priority[r]))
        return best
    pivot["role_primary"] = pivot.apply(primary_role, axis=1)

    # Identify metric columns to average across sessions
    metric_cols = [c for c in person_session_df.columns if c.startswith("ctx_") or c.startswith("p_") or c.startswith("ann_")]
    if metric_cols:
        agg = (person_session_df
               .groupby(["person_name","conference","year"], as_index=False)[metric_cols]
               .mean(numeric_only=True))
        out = pivot.merge(agg, on=["person_name","conference","year"], how="left")
    else:
        out = pivot

    # Team outcomes from person_to_team (across the whole dataset; if needed per conf-year, adjust mapping)
    team_rows = []
    for pname in out["person_name"]:
        lst = person_to_team.get(pname, [])
        funded = sum(1 for t in lst if t.get("funded_status", 0) == 1)
        unfund = sum(1 for t in lst if t.get("funded_status", 0) == 0)
        team_rows.append((pname, funded, unfund, ", ".join([t.get("team_id","") for t in lst])))
    team_df = pd.DataFrame(team_rows, columns=["person_name","teams_funded","teams_unfunded","team_ids"])
    team_df["teams_total"] = team_df["teams_funded"] + team_df["teams_unfunded"]

    out = out.merge(team_df, on="person_name", how="left")

    return out

# ---- DRIVER: iterate conferences, build combined outputs ----
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    core = conf_path / f"{conf_name}_session_outcomes.json"
    if not core.exists():
        continue

    year, conference, outcome, person_to_team, session_outcomes, features = load_conference_data(conf_path)

    ps = build_person_session(year, conference, conf_path, session_outcomes, features)
    py = build_person_year(ps, person_to_team)

    if not ps.empty:
        ALL_PERSON_SESSION.append(ps)
    if not py.empty:
        ALL_PERSON_YEAR.append(py)

# ---- COMBINE to single files ----
if ALL_PERSON_YEAR:
    all_py = (pd.concat(ALL_PERSON_YEAR, ignore_index=True)
                .sort_values(["person_name","year","conference"]))

    # --- add role flags for contrasts ---
    all_py["role_facilitator"]    = (all_py["role_primary"] == "facilitator").astype(int)
    all_py["role_nonfacilitator"] = 1 - all_py["role_facilitator"]

    all_py["teams_total"]  = all_py["teams_total"].fillna(0)
    all_py["teams_funded"] = all_py["teams_funded"].fillna(0)

    all_py["role_on_team"]   = (all_py["teams_total"]  > 0).astype(int)
    all_py["role_in_funded"] = (all_py["teams_funded"] > 0).astype(int)

    all_py["role_member"]      = (all_py["role_primary"] == "member").astype(int)
    all_py["role_participant"] = (all_py["role_primary"] == "participant").astype(int)

    out_path = OUTPUT_DIR / "ALL_person_year.csv"
    all_py.to_csv(out_path, index=False)
    print(f"Wrote ONE combined file: {out_path}  ({len(all_py)} rows)")
else:
    print("No person-year rows produced.")

# Optional: combined person-session file
# if ALL_PERSON_SESSION:
#     all_ps = (pd.concat(ALL_PERSON_SESSION, ignore_index=True)
#                 .sort_values(["person_name","year","conference","session_id"]))
#     out_path_ps = OUTPUT_DIR / "ALL_person_session.csv"
#     all_ps.to_csv(out_path_ps, index=False)
#     print(f"Wrote combined person-session file: {out_path_ps}  ({len(all_ps)} rows)")
# ==================== end ====================

## Regression Analysis

Figuring out the question: 

**By analyzing at the individual level, do facilitators act differently than team members?**

Also, considering again, the classification from above:

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

# Running the features alone (without existing knowledge)

Determining if people are (or not) facilitator, funded-team, on-team.

**Probably would just run cross-validation on this.**

In [2]:
# ==================== ALL-IN-ONE: regressions + CV classification + graphs ====================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# -------- CONFIG --------
CSV_PATH = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR  = Path("/Users/maxchalekson/Desktop/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_CONTROLS = True          # add sessions_total + ctx_* to regressions
INCLUDE_CTX_IN_CV = True     # include ctx_* alongside p_* and ann_* in classifiers
RANDOM_STATE = 42
N_SPLITS = 5

# ==================== LOAD ====================
df = pd.read_csv(CSV_PATH)

# ==================== PART A: DESCRIPTIVE REGRESSIONS ====================
import statsmodels.formula.api as smf

def _is_num_and_nonempty(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s) and s.notna().any()

# Outcomes
p_outcomes          = [c for c in df.columns if c.startswith("p_") and _is_num_and_nonempty(df[c])]
ann_count_outcomes  = [c for c in df.columns if c.startswith("ann_") and c.endswith("_count") and _is_num_and_nonempty(df[c])]
ann_mean_outcomes   = [c for c in df.columns if c.startswith("ann_") and c.endswith("_mean_score") and _is_num_and_nonempty(df[c])]

role_flags = ["role_facilitator","role_on_team","role_in_funded"]
for r in role_flags:
    if r not in df.columns:
        raise ValueError(f"Missing role flag: {r}")

# Controls
controls = []
if USE_CONTROLS:
    if "sessions_total" in df.columns and _is_num_and_nonempty(df["sessions_total"]):
        controls.append("sessions_total")
    # add all numeric ctx_* as controls
    ctx_cols = [c for c in df.columns if c.startswith("ctx_") and _is_num_and_nonempty(df[c])]
    controls += ctx_cols

def build_formula(y, x, controls_list):
    rhs = [x] + (controls_list or [])
    return f"{y} ~ " + " + ".join(rhs)

reg_rows = []
summaries_txt = []

for contrast in role_flags:
    for outcome in (p_outcomes + ann_count_outcomes + ann_mean_outcomes):
        needed = [outcome, contrast] + controls
        d = df[needed].dropna().copy()
        if len(d) < 25:  # tiny samples are unstable; skip quietly
            continue
        fml = build_formula(outcome, contrast, controls)
        model = smf.ols(fml, data=d).fit()
        robust = model.get_robustcov_results(cov_type="HC3")

        # safe param extraction
        names  = robust.model.exog_names
        params = dict(zip(names, robust.params))
        ses    = dict(zip(names, robust.bse))
        pvals  = dict(zip(names, robust.pvalues))

        coef = params.get(contrast, np.nan)
        se   = ses.get(contrast, np.nan)
        pval = pvals.get(contrast, np.nan)

        reg_rows.append({
            "outcome": outcome,
            "contrast": contrast,
            "n_used": len(d),
            "controls": ", ".join(controls) if controls else "(none)",
            "coef_role": coef,
            "se_role": se,
            "p_role": pval,
            "r2": robust.rsquared,
            "r2_adj": robust.rsquared_adj,
            "formula": fml,
        })
        summaries_txt.append(f"=== {contrast} :: {outcome} ===\n{robust.summary().as_text()}\n")

# Save regression outputs
reg_df = pd.DataFrame(reg_rows).sort_values(["contrast","outcome"])
reg_df.to_csv(OUT_DIR / "regression_tidy_results.csv", index=False)
with open(OUT_DIR / "regression_model_summaries.txt", "w") as f:
    f.write("\n\n".join(summaries_txt))

# ==================== PART B: PREDICTION (CV Logistic Regression) ====================
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

TARGETS = [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Unfunded"),
]

# Exclude leaky / ID columns from features
LEAKY_OR_ID = {
    "person_name","conference","year",
    "facilitator","member","participant","unknown","sessions_total",
    "role_primary","role_facilitator","role_nonfacilitator",
    "role_on_team","role_in_funded","role_member","role_participant",
    "teams_total","teams_funded","teams_unfunded","team_ids"
}

p_feats   = [c for c in df.columns if c.startswith("p_")   and _is_num_and_nonempty(df[c])]
ann_feats = [c for c in df.columns if c.startswith("ann_") and _is_num_and_nonempty(df[c])]
ctx_feats = [c for c in df.columns if c.startswith("ctx_") and _is_num_and_nonempty(df[c])] if INCLUDE_CTX_IN_CV else []
feat_cols = [c for c in (p_feats + ann_feats + ctx_feats) if c not in LEAKY_OR_ID]

# Drop zero-variance
feat_cols = [c for c in feat_cols if df[c].std(skipna=True) > 0]
if not feat_cols:
    raise ValueError("No usable behavior features (p_* / ann_* / ctx_*) after cleaning.")

pd.Series(feat_cols, name="feature").to_csv(OUT_DIR / "clf_features_used.csv", index=False)

num_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0.0)),
                     ("scale", StandardScaler())])
pre = ColumnTransformer([("num", num_pipe, feat_cols)], remainder="drop")
logit = LogisticRegression(penalty="l2", solver="liblinear", class_weight="balanced",
                           max_iter=2000, random_state=RANDOM_STATE)
clf = Pipeline([("prep", pre), ("clf", logit)])
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def plot_roc(y_true, y_prob, title, outpath):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0,1],[0,1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)
    return roc_auc

clf_rows = []
for target, desc in TARGETS:
    dsub = df.dropna(subset=[target]).copy()
    if dsub[target].nunique() < 2:
        continue
    y = dsub[target].astype(int)
    X = dsub[feat_cols]

    scoring = {"roc_auc":"roc_auc", "accuracy":"accuracy", "precision":"precision", "recall":"recall", "f1":"f1"}
    cv_res = cross_validate(clf, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    metrics = {m: (cv_res[f"test_{m}"].mean(), cv_res[f"test_{m}"].std()) for m in scoring}

    # Held-out predictions for ROC + report
    y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
    y_hat  = (y_prob >= 0.5).astype(int)
    report = classification_report(y, y_hat, output_dict=True, zero_division=0)
    pd.DataFrame(report).to_csv(OUT_DIR / f"clf_classification_report_{target}.csv")

    roc_png = OUT_DIR / f"clf_roc_{target}.png"
    _ = plot_roc(y, y_prob, f"ROC – {desc}", roc_png)

    # Fit once on all data for coefficients
    clf.fit(X, y)
    coefs = pd.Series(clf.named_steps["clf"].coef_.ravel(), index=feat_cols).sort_values(ascending=False)
    coefs.to_csv(OUT_DIR / f"clf_top_coefs_{target}.csv")

    clf_rows.append({
        "target": target, "description": desc,
        "n_rows": int(len(y)), "pos_count": int(y.sum()), "neg_count": int((1-y).sum()),
        "roc_auc_mean": metrics["roc_auc"][0], "roc_auc_sd": metrics["roc_auc"][1],
        "accuracy_mean": metrics["accuracy"][0], "accuracy_sd": metrics["accuracy"][1],
        "precision_mean": metrics["precision"][0], "precision_sd": metrics["precision"][1],
        "recall_mean": metrics["recall"][0], "recall_sd": metrics["recall"][1],
        "f1_mean": metrics["f1"][0], "f1_sd": metrics["f1"][1],
        "roc_curve_png": str(roc_png)
    })

pd.DataFrame(clf_rows).sort_values("target").to_csv(OUT_DIR / "clf_results_summary.csv", index=False)

# ==================== PART C: GRAPHS (boxplots + mean±CI for key features) ====================
def mean_ci(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    n = len(s)
    if n == 0:
        return np.nan, np.nan
    m = s.mean()
    se = s.std(ddof=1)/np.sqrt(n) if n>1 else 0
    return m, 1.96*se

def subset_for_contrast(df0, contrast):
    # Keep it simple: no filtering; show all people for each contrast (0 vs 1)
    return df0.dropna(subset=[contrast])

labels = {
    "role_facilitator": ("Non-facilitator","Facilitator"),
    "role_on_team": ("Not on team","On team"),
    "role_in_funded": ("Unfunded team","Funded team"),
}

# Choose a small, interpretable set to plot (add more if you like)
plot_p = ["p_speaking_duration_sec","p_turns","p_interruptions_made","p_screenshare_segments"]
plot_ann = []
# Prefer common annotation codes if present
for cand in [
    "ann_knowledge_sharing_count","ann_coordination_and_decision_practices_count",
    "ann_coordination_decision_practices_count","ann_evaluation_practices_count",
    "ann_participation_dynamics_count","ann_relational_climate_count"
]:
    if cand in df.columns:
        plot_ann.append(cand)

def boxplot_two_groups(d, y, g, xlabels, fname):
    dd = d[[y,g]].dropna()
    if dd.empty: return
    data0 = dd.loc[dd[g]==0, y].values
    data1 = dd.loc[dd[g]==1, y].values
    fig, ax = plt.subplots(figsize=(6,4.5))
    ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
    ax.set_title(f"{y} by {g}")
    ax.set_ylabel(y)
    ax.set_xlabel(g)
    ax.grid(True, linestyle="--", alpha=0.4)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.close(fig)

def bar_meanci_two_groups(d, y, g, xlabels, fname):
    dd = d[[y,g]].dropna()
    if dd.empty: return
    m0, c0 = mean_ci(dd.loc[dd[g]==0, y])
    m1, c1 = mean_ci(dd.loc[dd[g]==1, y])
    fig, ax = plt.subplots(figsize=(6,4.5))
    ax.bar([0,1], [m0,m1], yerr=[c0,c1], capsize=6)
    ax.set_xticks([0,1]); ax.set_xticklabels(xlabels)
    ax.set_title(f"Mean {y} (±95% CI) by {g}")
    ax.set_ylabel(y)
    ax.set_xlabel(g)
    ax.grid(True, linestyle="--", alpha=0.4, axis="y")
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.close(fig)

for contrast in role_flags:
    dsub = subset_for_contrast(df, contrast)
    xlabels = labels[contrast]

    # p_*: use boxplots for continuous-ish, bar CI for counts
    if "p_speaking_duration_sec" in plot_p and "p_speaking_duration_sec" in df.columns:
        boxplot_two_groups(dsub, "p_speaking_duration_sec", contrast, xlabels,
                           f"box_p_speaking_duration_sec_{contrast}.png")
    if "p_turns" in plot_p and "p_turns" in df.columns:
        boxplot_two_groups(dsub, "p_turns", contrast, xlabels,
                           f"box_p_turns_{contrast}.png")
    for c in ["p_interruptions_made","p_screenshare_segments"]:
        if c in plot_p and c in df.columns:
            bar_meanci_two_groups(dsub, c, contrast, xlabels, f"bar_{c}_{contrast}.png")

    # ann_* counts: bar CI makes sense (means across people)
    for c in plot_ann:
        bar_meanci_two_groups(dsub, c, contrast, xlabels, f"bar_{c}_{contrast}.png")

print(f"\nDone. Outputs in: {OUT_DIR}")
print("Files written include:")
print(" - regression_tidy_results.csv")
print(" - regression_model_summaries.txt")
print(" - clf_results_summary.csv")
print(" - clf_features_used.csv")
print(" - clf_classification_report_<target>.csv")
print(" - clf_top_coefs_<target>.csv")
print(" - clf_roc_<target>.png")
print(" - box_/bar_*.png for key p_* and ann_* features across contrasts")
# ==================== END ====================

/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42719/270766337.py:228: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42719/270766337.py:228: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42719/270766337.py:228: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42


Done. Outputs in: /Users/maxchalekson/Desktop/outputs
Files written include:
 - regression_tidy_results.csv
 - regression_model_summaries.txt
 - clf_results_summary.csv
 - clf_features_used.csv
 - clf_classification_report_<target>.csv
 - clf_top_coefs_<target>.csv
 - clf_roc_<target>.png
 - box_/bar_*.png for key p_* and ann_* features across contrasts
